
# Cuando el camino más corto no es el mejor

Ruta de evacuación en una zona inundada 
Equipo: CafeSmart
Integrantes:
Morales Mendez Shryley Nicoles
Jimenez Andrade Monserrat
Tejero Hernandez Rubi
Sarao Morales Rubi

Un agente debe ir de un punto inicial a un refugio sobre un terreno con celdas secas, de inundación leve / media / alta (transitables pero riesgosas) y bloqueadas. Se comparan cuatro estrategias (BFS, Dijkstra, A*, Voraz) con ejecuciones reales.

Índice: 
· 1 Modelo 
· 2 Algoritmos 
· 3 Validador 
· 4 Escenario y visualización
 · 5 Comparación 
 · 6 Pruebas (T1–T5) 
 · 7 Prueba adicional (T6) 
 · 8 Decisiones y declaración de IA

## 1. Modelo del terreno
Cada celda tiene un nivel: 0 seco, 1 leve, 2 media, 3 alta, 4 bloqueado.

- **Bloqueo = restricción dura:** las celdas nivel 4 nunca se expanden.
- **Riesgo = costo blando:** costo de entrar a una celda = `1 + λ · riesgo(nivel)`, con riesgo 0, 1, 3, 10. El "1" es el esfuerzo de dar un paso; el resto penaliza el peligro. La penalización no es lineal para que una celda alta pese más que varias leves.
- **Movimiento en 4 direcciones** (sin diagonales, para evitar "cortar esquinas" entre bloqueos).
- **Costos ≥ 1**, así la distancia Manhattan nunca sobreestima el costo restante (heurística admisible para A*).

In [ ]:
import heapq, time, random
from collections import deque

# Niveles de terreno
DRY, LOW, MID, HIGH, BLOCKED = 0, 1, 2, 3, 4
# Penalización de riesgo por entrar a una celda (sumada al costo base 1)
RISK_W = {DRY: 0, LOW: 1, MID: 3, HIGH: 10}
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 4-conectividad


def cell_cost(level, lam=1.0, w=RISK_W):
    """Costo de ENTRAR a una celda: 1 + lam * riesgo. None si bloqueada."""
    if level == BLOCKED:
        return None
    return 1.0 + lam * w[level]


def neighbors(grid, pos):
    r, c = pos
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != BLOCKED:
            yield (nr, nc)


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def path_cost(grid, path, lam=1.0, w=RISK_W):
    return sum(cell_cost(grid[r][c], lam, w) for r, c in path[1:])


def _rebuild(parent, goal):
    p, cur = [], goal
    while cur is not None:
        p.append(cur)
        cur = parent[cur]
    return p[::-1]


def _result(grid, path, expanded, pushed, t0, lam, w):
    return {
        "path": path,
        "cost": None if path is None else path_cost(grid, path, lam, w),
        "steps": None if path is None else len(path) - 1,
        "expanded": expanded,
        "generated": pushed,
        "time_ms": (time.perf_counter() - t0) * 1000,
    }

## 4. Escenario base

Un muro bloqueado (columnas 6–8) divide el mapa. Hay un **atajo** por la fila 7 con 3 celdas de inundación alta y un **rodeo seco** por la fila 14. Inicio (7,1), refugio (7,14) (fila, columna).

In [ ]:
# ---------------- Escenarios ----------------
def base_scenario():
    """16x16. Muro inundado (bloqueado) en cols 6-8 con dos pasos:
    atajo por fila 7 (riesgo ALTO) y rodeo seco por fila 14."""
    n = 16
    g = [[DRY] * n for _ in range(n)]
    for r in range(n):
        for c in (6, 7, 8):
            g[r][c] = BLOCKED
    for c in (6, 7, 8):
        g[7][c] = HIGH       # atajo peligroso
        g[14][c] = DRY       # rodeo seguro
    # obstaculo realista: zonas de inundación leve/media
    for r in range(2, 5):
        for c in range(1, 4):
            g[r][c] = LOW
    for r in range(10, 13):
        for c in range(10, 14):
            g[r][c] = MID
    for r in range(3, 6):
        for c in range(11, 14):
            g[r][c] = LOW
    return g, (7, 1), (7, 14)


def random_grid(n, rng, p_block=0.2, p_risk=0.35):
    g = [[DRY] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            x = rng.random()
            if x < p_block: g[r][c] = BLOCKED
            elif x < p_block + p_risk: g[r][c] = rng.choice([LOW, MID, HIGH])
    g[0][0] = DRY; g[n - 1][n - 1] = DRY
    return g